In [0]:
-- ══════════════════════════════════════════════════════════════════════════════
-- MODULAR PIPELINE: Incrementality vs Cannibalization — step-by-step checkpoints
-- Run this whole block; the final SELECT shows a row per pipeline stage.
-- ══════════════════════════════════════════════════════════════════════════════

WITH
-- ── STEP 1: Send list ────────────────────────────────────────────────────────
sent AS (
    SELECT DISTINCT f.message_profile_id, e.CDE_ID
    FROM cpx_dataanalytics_gold.marketing.mkt_adobe_message_feedback f
    LEFT JOIN cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address e
           ON e.Email_Address = f.Email_Address
    WHERE f.journey_id = '96b494ce-f57e-4068-b24c-049fd5684fee'
      AND f.journey_action_id IN (
            'ef8134cd-3dd9-4553-b8cf-c273db9d84b9',
            'c6ba2ed4-5814-417d-8d46-281932defbc5')
      AND f.feedback_status = 'sent'
      AND e.CDE_ID IS NOT NULL
),

-- ── STEP 2: Opens ────────────────────────────────────────────────────────────
opens AS (
    SELECT DISTINCT message_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_adobe_email_tracking
    WHERE interaction_type = 'open'
      AND CAST(timestamp AS DATE) <= DATE '2026-08-26'
),

-- ── STEP 3: Openers (dedup fix — any open across sends = opener) ────────────
openers AS (
    SELECT s.CDE_ID
    FROM sent s
    JOIN opens o USING (message_profile_id)
    GROUP BY s.CDE_ID
),

-- ── STEP 4: Eligibility filter ───────────────────────────────────────────────
mm_txn_keys AS (
    SELECT DISTINCT t.CDE_ID, t.Transaction_Number, t.LocationId
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_film fm ON fm.film_id = t.film_id
    WHERE t.SCENEFLAG  = 'Yes'
      AND t.TicketFLAG = 'Yes'
      AND t.CDE_ID IS NOT NULL
      AND t.VisitDateId BETWEEN 20260702 AND 20260821
      AND fm.title_id  = '38680'
),
ineligible_members AS (
    SELECT DISTINCT k.CDE_ID
    FROM mm_txn_keys k
    JOIN cpx_dataanalytics_gold.customer360.c360_f_transaction t
      ON t.Transaction_Number = k.Transaction_Number
     AND t.CDE_ID              = k.CDE_ID
     AND t.LocationId          = k.LocationId
    WHERE t.SCENE_Points_Redeemed > 0
),

-- ── STEP 5: Slim transaction base ───────────────────────────────────────────
trans_slim AS (
    SELECT
        t.CDE_ID, dt.datefull,
        loc.LC_Location_Type_Description  AS loc_type,
        loc.LC_Trade_Brand_Description    AS brand,
        fm.title_id, t.TicketFLAG, t.LocationId, t.Net_Revenue,
        CONCAT_WS('|', CAST(t.VisitDateId AS STRING),
                       CAST(t.LocationId  AS STRING), t.CDE_ID) AS visit_key
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_date dt ON dt.dateid = t.VisitDateId
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON loc.LocationID = t.LocationId
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_film fm ON fm.film_id = t.film_id
    WHERE t.SCENEFLAG = 'Yes'
      AND dt.datefull BETWEEN DATE '2026-07-02' AND DATE '2026-09-11'
      AND (
            loc.LC_Location_Type_Description = 'Cineplex Theatre'
         OR (loc.LC_Location_Type_Description = 'Restaurant/Store/Games'
             AND loc.LC_Trade_Brand_Description IN ('Rec Room','Playdium'))
          )
),

-- ── STEP 6: M&M cinema tickets, cohort assigned ─────────────────────────────
cinema_tickets AS (
    SELECT DISTINCT
        ts.CDE_ID,
        CASE WHEN o.CDE_ID IS NOT NULL THEN 'opener' ELSE 'control' END AS cohort,
        ts.datefull AS cinema_date
    FROM trans_slim ts
    LEFT JOIN openers o ON o.CDE_ID = ts.CDE_ID
    LEFT JOIN ineligible_members ie ON ie.CDE_ID = ts.CDE_ID
    WHERE ts.title_id  = '38680'
      AND ts.datefull BETWEEN DATE '2026-07-02' AND DATE '2026-08-21'
      AND ts.loc_type   = 'Cineplex Theatre'
      AND ts.TicketFLAG = 'Yes'
      AND ie.CDE_ID IS NULL
),

-- ── STEP 7: LBE visits ───────────────────────────────────────────────────────
lbe_visits AS (
    SELECT CDE_ID, datefull AS lbe_date, visit_key AS lbe_visit_key,
           SUM(Net_Revenue) AS lbe_revenue
    FROM trans_slim
    WHERE loc_type = 'Restaurant/Store/Games'
    GROUP BY CDE_ID, datefull, visit_key
),

-- ── STEP 8: First LBE within 21 days per ticket (same-day included) ────────
ticket_attribution AS (
    SELECT /*+ RANGE_JOIN(l, 21) */
        c.CDE_ID, c.cohort, c.cinema_date,
        l.lbe_date, l.lbe_visit_key, l.lbe_revenue,
        ROW_NUMBER() OVER (PARTITION BY c.CDE_ID, c.cohort, c.cinema_date
                           ORDER BY l.lbe_date NULLS LAST) AS lbe_rank
    FROM cinema_tickets c
    LEFT JOIN lbe_visits l
      ON  l.CDE_ID   = c.CDE_ID
     AND  l.lbe_date >= c.cinema_date
     AND  l.lbe_date <= DATE_ADD(c.cinema_date, 21)
),
first_lbe_per_ticket AS (
    SELECT CDE_ID, cohort, cinema_date, lbe_date, lbe_visit_key, lbe_revenue
    FROM ticket_attribution
    WHERE lbe_rank = 1
),

-- ── STEP 9: Per-member stats (payout cap applied) ───────────────────────────
member_stats AS (
    SELECT cohort, CDE_ID,
           COUNT(*) AS total_tickets,
           COUNT(lbe_date) AS qualifying_tickets,
           LEAST(COUNT(lbe_date), 10) AS payout_events
    FROM first_lbe_per_ticket
    GROUP BY cohort, CDE_ID
),
member_revenue AS (
    SELECT cohort, CDE_ID, SUM(lbe_revenue) AS attributed_revenue
    FROM (
        SELECT DISTINCT cohort, CDE_ID, lbe_visit_key, lbe_revenue
        FROM first_lbe_per_ticket
        WHERE lbe_visit_key IS NOT NULL
    )
    GROUP BY cohort, CDE_ID
),

-- ── STEP 10: Cohort rollup ───────────────────────────────────────────────────
cohort_stats AS (
    SELECT ms.cohort,
           COUNT(DISTINCT ms.CDE_ID) AS total_viewers,
           SUM(CASE WHEN ms.qualifying_tickets > 0 THEN 1 ELSE 0 END) AS cross_visitors,
           SUM(ms.payout_events) AS payout_events,
           COALESCE(SUM(mr.attributed_revenue), 0) AS attributed_revenue
    FROM member_stats ms
    LEFT JOIN member_revenue mr USING (cohort, CDE_ID)
    GROUP BY ms.cohort
),
cohort_rates AS (
    SELECT cohort, total_viewers, cross_visitors, payout_events, attributed_revenue,
           attributed_revenue / total_viewers AS rev_per_viewer
    FROM cohort_stats
),

-- ── STEP 11: Waterfall inputs ────────────────────────────────────────────────
waterfall_inputs AS (
    SELECT
        MAX(CASE WHEN cohort='opener' THEN total_viewers END)   AS opener_viewers,
        MAX(CASE WHEN cohort='opener' THEN rev_per_viewer END)  AS opener_rev_per_viewer,
        MAX(CASE WHEN cohort='opener' THEN payout_events END)   AS opener_payout_events,
        MAX(CASE WHEN cohort='control' THEN rev_per_viewer END) AS control_rev_per_viewer
    FROM cohort_rates
)

-- ══════════════════════════════════════════════════════════════════════════════
-- OUTPUT: one row per pipeline step, showing the count/metric at each stage
-- ══════════════════════════════════════════════════════════════════════════════
SELECT '1. Total sent (unique CDE_ID)'        AS step, CAST(COUNT(DISTINCT CDE_ID) AS STRING) AS value FROM sent
UNION ALL
SELECT '2. Total opens (unique message_profile_id)', CAST(COUNT(*) AS STRING) FROM opens
UNION ALL
SELECT '3. Openers (deduped, unique CDE_ID)', CAST(COUNT(*) AS STRING) FROM openers
UNION ALL
SELECT '4. Ineligible members (redeemed on M&M ticket)', CAST(COUNT(*) AS STRING) FROM ineligible_members
UNION ALL
SELECT '5. trans_slim row count', CAST(COUNT(*) AS STRING) FROM trans_slim
UNION ALL
SELECT '6a. M&M viewers — opener', CAST(COUNT(DISTINCT CDE_ID) AS STRING) FROM cinema_tickets WHERE cohort = 'opener'
UNION ALL
SELECT '6b. M&M viewers — control', CAST(COUNT(DISTINCT CDE_ID) AS STRING) FROM cinema_tickets WHERE cohort = 'control'
UNION ALL
SELECT '7. Total LBE visit-day records', CAST(COUNT(*) AS STRING) FROM lbe_visits
UNION ALL
SELECT '8a. Cross-visitors — opener', CAST(COUNT(DISTINCT CDE_ID) AS STRING) FROM first_lbe_per_ticket WHERE cohort = 'opener' AND lbe_date IS NOT NULL
UNION ALL
SELECT '8b. Cross-visitors — control', CAST(COUNT(DISTINCT CDE_ID) AS STRING) FROM first_lbe_per_ticket WHERE cohort = 'control' AND lbe_date IS NOT NULL
UNION ALL
SELECT '9a. Payout events — opener', CAST(SUM(payout_events) AS STRING) FROM member_stats WHERE cohort = 'opener'
UNION ALL
SELECT '9b. Payout events — control', CAST(SUM(payout_events) AS STRING) FROM member_stats WHERE cohort = 'control'
UNION ALL
SELECT '10a. Opener: viewers / cross-visitors / revenue / rev-per-viewer',
       CONCAT_WS(' | ', CAST(total_viewers AS STRING), CAST(cross_visitors AS STRING),
                 CAST(ROUND(attributed_revenue,2) AS STRING), CAST(ROUND(rev_per_viewer,2) AS STRING))
FROM cohort_rates WHERE cohort = 'opener'
UNION ALL
SELECT '10b. Control: viewers / cross-visitors / revenue / rev-per-viewer',
       CONCAT_WS(' | ', CAST(total_viewers AS STRING), CAST(cross_visitors AS STRING),
                 CAST(ROUND(attributed_revenue,2) AS STRING), CAST(ROUND(rev_per_viewer,2) AS STRING))
FROM cohort_rates WHERE cohort = 'control'
UNION ALL
SELECT '11a. Total opener revenue', CAST(ROUND(opener_viewers * opener_rev_per_viewer, 2) AS STRING) FROM waterfall_inputs
UNION ALL
SELECT '11b. Cannibalized revenue (control rate)', CAST(ROUND(opener_viewers * control_rev_per_viewer, 2) AS STRING) FROM waterfall_inputs
UNION ALL
SELECT '11c. Incremental revenue', CAST(ROUND(opener_viewers * (opener_rev_per_viewer - control_rev_per_viewer), 2) AS STRING) FROM waterfall_inputs
UNION ALL
SELECT '11d. Cost (points redeemed)', CAST(ROUND(opener_payout_events * 250 * 0.009237, 2) AS STRING) FROM waterfall_inputs
UNION ALL
SELECT '11e. Net incremental value',
       CAST(ROUND(opener_viewers * (opener_rev_per_viewer - control_rev_per_viewer)
                 - opener_payout_events * 250 * 0.009237, 2) AS STRING)
FROM waterfall_inputs;